<a href="https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane and why

I'm choosing **Lane 4: CTR/Engagement Opportunity Scoring**. In my Week 1 notebook, I
grouped CTR by content_type within the same position_tier and found that raw CTR
comparisons are misleading unless you account for both position and sample size. This
is exactly the core method Lane 4 uses: compare a page's CTR only to others in the same
position tier, then flag pages sitting below what's expected for their tier.

A quick check on the starter dataset backs this up: mean CTR ranges from 0.355% at
page_1 down to 0.055% at the deep tier, a roughly 6.5x spread, among the 22,006 pages
(73.4% of the dataset) with enough impressions (>=100) to trust the number. That's a
real, sizeable pattern worth 7 weeks of digging, and there's already more nuance inside
it than "higher position = higher CTR" (page_1 actually edges out top_3 slightly, which
is worth investigating further).

This lane also avoids the harder problem of defining a future decline/recovery label
right away, which fits well as I'm building my ML fundamentals for the first time,
coming from a data analysis background.


In [19]:
!git clone https://github.com/solasobambo-prog/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 138 (delta 47), reused 98 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.87 MiB | 11.18 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [20]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [21]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

visible = df[df["impressions_90d"] >= 100]
ctr_by_tier = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)

print("Mean CTR by position tier (impressions >= 100):")
print(ctr_by_tier.round(3))
print(f"\nPages with impressions_90d >= 100: {len(visible):,} of {len(df):,} total ({len(visible)/len(df):.1%})")
print(f"CTR spread across tiers: {ctr_by_tier.max():.3f} down to {ctr_by_tier.min():.3f}")

Mean CTR by position tier (impressions >= 100):
position_tier
page_1      0.355
top_3       0.334
striking    0.256
page_3_5    0.142
deep        0.055
Name: ctr, dtype: float64

Pages with impressions_90d >= 100: 22,006 of 30,000 total (73.4%)
CTR spread across tiers: 0.355 down to 0.055


## 2. The question: decision, action, cost of a wrong call

**Decision:** Which visible pages (getting real search impressions) should an editor
review first because they're under-capturing clicks relative to their position?

**Who acts, and how:** A content editor or SEO reviewer opens the flagged pages and
decides whether to rewrite the title/meta description, improve the snippet, or check
if the content actually matches what searchers want.

**Cost of a wrong call:**
- False positive: editor time spent reviewing a page that didn't actually need it.
- False negative: a real opportunity (a page ranking well but losing clicks it should
  be getting) goes unnoticed and keeps under-performing.

**Why not just a simple rule:** A fixed CTR threshold (e.g. "flag anything under 1%")
doesn't work, because expected CTR varies a lot by position tier (0.355% at page_1
down to 0.055% at deep, from my Section 1 numbers). A page needs to be compared only
to others in its own tier, and volume has to be accounted for too (Week 1 showed
sample sizes as small as 1-5 rows producing misleading averages). That's more than one
clean if-statement can safely handle.

**One-paragraph frame:** For a content editor deciding which pages to review first,
we will build a ranked list from the starter/warehouse search data, scoring pages by
how far their CTR falls below the expected CTR for their position tier, adjusted for
volume. A wrong call costs either wasted review time or a missed click-opportunity.
A plain threshold isn't enough because "low CTR" only makes sense relative to position
and sample size. We will claim only observed, directional, decision-support results,
not proof that any specific fix will work.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [23]:
# Section 3: Quick look at the data — evidence for Lane 4

# Number 1 & 2: CTR range across position tiers (from Section 1)
visible = df[df["impressions_90d"] >= 100]
ctr_by_tier = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("Mean CTR by position tier (impressions >= 100):")
print(ctr_by_tier.round(3))
print(f"\nCTR spread across tiers: {ctr_by_tier.max():.3f}% down to {ctr_by_tier.min():.3f}% (~{ctr_by_tier.max()/ctr_by_tier.min():.1f}x)")

# Number 3: how many pages sit below their own tier's expected CTR (rough opportunity size)
visible = visible.copy()
visible["tier_avg_ctr"] = visible["position_tier"].map(ctr_by_tier)
below_expected = visible[visible["ctr"] < visible["tier_avg_ctr"]]
print(f"\nPages below their tier's average CTR: {len(below_expected):,} of {len(visible):,} ({len(below_expected)/len(visible):.1%})")


Mean CTR by position tier (impressions >= 100):
position_tier
page_1      0.355
top_3       0.334
striking    0.256
page_3_5    0.142
deep        0.055
Name: ctr, dtype: float64

CTR spread across tiers: 0.355% down to 0.055% (~6.4x)

Pages below their tier's average CTR: 14,797 of 22,006 (67.2%)


Three numbers support this lane:

1. Mean CTR varies from 0.355% (page_1) down to 0.055% (deep) across position tiers,
   a roughly 6.4x spread, confirming position tier matters a lot.
2. 22,006 of 30,000 pages (73.4%) have enough impressions (>=100) to trust their CTR
   number, so the lane has a large usable base, not just a handful of pages.
3. Of those, 14,797 (67.2%) sit below their own tier's average CTR, a large pool of
   "under-capturing" candidates. That's expected in any distribution with an average
   (roughly half will always be below the mean), so this number alone doesn't prove
   opportunity, it just confirms there's enough spread and volume within tiers to build
   a meaningful ranking on top of.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful words: what I can and can't claim

**What my work CAN say:**
- Observed patterns: e.g. "pages in this dataset show CTR varying by position tier,
  from 0.355% at page_1 down to 0.055% at deep."
- Directional signals: e.g. "pages sitting below their tier's expected CTR are more
  likely to represent an under-capturing opportunity, based on this sample."
- Decision-support output: a ranked list of pages an editor could review first, with
  reason codes explaining why each page was flagged (e.g. "high impressions, CTR below
  tier average").

**What my work CANNOT say:**
- That any specific page's low CTR is CAUSED by a bad title, meta description, or
  content quality; I have no experiment, only observed data, so I can't prove cause.
- That fixing a flagged page WILL improve its CTR that would require testing a
  before/after change, which this project doesn't do.
- Anything about how Google's ranking algorithm works, my data shows what happened
  in search results, not why Google's algorithm produced that outcome.
- That a page's low CTR compared to its tier average is definitely a "problem",
  since roughly half of pages will sit below any average by definition, being below
  average isn't automatically a real issue on its own (I noted this in Section 3 too).

**In short:** this project ranks candidates for human review based on observed,
tier-adjusted patterns. It does not prove why those patterns exist, and it does not
guarantee that acting on a recommendation will produce a specific result.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.